# Exploration des données Yu-Gi-Oh!
Ce notebook explore la base SQLite créée par `scripts/init_db.py`.

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect('../data/yugioh.db')

# Chargement des tables
cards  = pd.read_sql('SELECT * FROM cards', con)
sets   = pd.read_sql('SELECT * FROM card_sets', con)
prices = pd.read_sql('SELECT * FROM card_prices', con)

print(f'Cartes  : {len(cards)}')
print(f'Sets    : {len(sets)}')
print(f'Prix    : {len(prices)}')

## 1. Vue d'ensemble du dataset

In [ ]:
cards.head(3)

In [ ]:
cards.dtypes

## 2. Distribution des types de cartes

In [ ]:
cards['type'].value_counts()

## 3. Attributs des monstres (DARK, LIGHT, FIRE...)

In [ ]:
cards['attribute'].value_counts()

## 4. Top 20 archetypes (nombre de cartes)

In [ ]:
cards['archetype'].value_counts().head(20)

## 5. Banlist TCG — cartes bannies, limitées, semi-limitées

In [ ]:
banlist = cards[cards['ban_tcg'].notna()][['name', 'type', 'archetype', 'ban_tcg']]
banlist['ban_tcg'].value_counts()

In [ ]:
# Cartes bannies
banlist[banlist['ban_tcg'] == 'Banned'][['name', 'archetype']].reset_index(drop=True)

## 6. Distribution ATK/DEF des monstres

In [ ]:
monsters = cards[cards['atk'].notna()]
print('ATK moyenne :', round(monsters['atk'].mean(), 0))
print('ATK max     :', monsters['atk'].max())
print()
print('DEF moyenne :', round(monsters['def'].mean(), 0))
print('DEF max     :', monsters['def'].max())

## 7. Prix — cartes les plus chères (Cardmarket)

In [ ]:
df = cards.merge(prices, left_on='id', right_on='card_id')
df[['name', 'archetype', 'cardmarket_price']].sort_values('cardmarket_price', ascending=False).head(10)

## 8. Archetypes les plus représentés dans les sets

In [ ]:
# Nombre d'apparitions en booster par archetype
merged = sets.merge(cards[['id', 'archetype']], left_on='card_id', right_on='id')
merged.groupby('archetype').size().sort_values(ascending=False).head(15)